## 96. Global Coverage

### 1. Packages

In [2]:
# Packages
import geopandas as gpd
import glob
import os
import rioxarray as rxr
from tqdm import tqdm

C:\Users\white_rn\AppData\Local\Temp\ipykernel_10116\1928037731.py:2: UserWarning: Shapely 2.0 is installed, but because PyGEOS is also installed, GeoPandas will still use PyGEOS by default for now. To force to use and test Shapely 2.0, you have to set the environment variable USE_PYGEOS=0. You can do this before starting the Python process, or in your code before importing geopandas:

import os
os.environ['USE_PYGEOS'] = '0'
import geopandas

In a future release, GeoPandas will switch to using Shapely by default. If you are using PyGEOS directly (calling PyGEOS functions on geometries from GeoPandas), this will then stop working and you are encouraged to migrate from PyGEOS to Shapely 2.0 (https://shapely.readthedocs.io/en/latest/migration_pygeos.html).
  import geopandas as gpd


### 3. Settings

In [3]:
# Settings
mode = 'intertidal_improved_100m_global'   # Specify mode, either 'intertidal' or 'subtidal'
start_date = '2021-01-01'                  # Start date of the composites
stop_date = '2022-01-01'                   # End date of the composites
compo_int = 12                             # Composite interval [months]
compo_len = 12                             # Composite length [months]
scale = 100                                # Output resolution of the image [m]
crs = 'EPSG:4326'                          # Output projection of the image

upscale = 100                              # Upscaling factor for the image
res_arc_min = 1/16                         # resolution of the image in arc minutes
res_deg = res_arc_min / 60                 # resolution of the image in degrees


# Tiling (see https://www.openearth.nl/rws-bathymetry/2019.html)
zoom_levels = [9, 10, 11] # list with zoom levels
zoom_level = zoom_levels[1] # zoom level to be used

# Directories and files
dir_path_base = r'p:\11209821-cmems-global-sdb'
file_path_tiles = os.path.join(dir_path_base, '00_miscellaneous', 'AOI_polygons_world', f'df_boxes_world_Z{zoom_level}_filtered_v2.parquet') # Tiles file
file_path_sdbs = glob.glob(os.path.join(dir_path_base, '01_intertidal', '02_data', '05_calibrated', mode, '05_reprojected', '*.tif'))        # SDB files
file_path_sdbs = [file_path for file_path in file_path_sdbs if not file_path.endswith('.tif.aux.xml')]
print('Number of SDB files:', len(file_path_sdbs))

Number of SDB files: 2235


### 4. Read tiles

In [4]:
# Read tiles
gdf_tiles = gpd.read_parquet(file_path_tiles)
gdf_tiles['nearest_station_distance'] = gdf_tiles['nearest_station_distance']/1000  # Convert to km
gdf_tiles['intertidal_coverage'] = gdf_tiles['intertidal_coverage']*100  # Convert to percentage
gdf_tiles['intertidal_coverage_ed'] = gdf_tiles['intertidal_coverage_ed']*100  # Convert to percentage
gdf_tiles['processed'] = (gdf_tiles['nearest_station_distance'] < 37) & (gdf_tiles['intertidal_coverage'] > 0) & (gdf_tiles['intertidal_coverage_ed'] > 1)
gdf_tiles = gdf_tiles[gdf_tiles['processed']]

# Add sdb file_path to tiles
for i, row in tqdm(gdf_tiles.iterrows(), total=gdf_tiles.shape[0]):
    file_path = [file_path for file_path in file_path_sdbs if os.path.basename(file_path).startswith(row['name'])]
    gdf_tiles.at[i, 'file_path'] = file_path[0] if file_path else None

# Remove tiles without SDB file_path
gdf_tiles = gdf_tiles[gdf_tiles['file_path'].notna()]

100%|██████████| 11332/11332 [04:20<00:00, 43.43it/s]


### 5. Calculate area based on feasability map

In [5]:
# Reproject tiles to metres
gdf_tiles = gdf_tiles.to_crs('EPSG:3857')

# Calculate area of the tiles
gdf_tiles['tile_area'] = gdf_tiles['geometry'].area / 1e6  # Convert to km2

# Calculate coverage
gdf_tiles['intertidal_area_ed'] = gdf_tiles['tile_area'] * gdf_tiles['intertidal_coverage_ed']/100

### 6. Calculate area based on sdb results

In [23]:
# Reproject tiles to WGS84
gdf_tiles = gdf_tiles.to_crs('EPSG:4326')

# Get area of the sdb files
for i, row in tqdm(gdf_tiles.iterrows(), total=gdf_tiles.shape[0]):
    # Read sdb file
    ds = rxr.open_rasterio(row['file_path'])
    
    # Clip to tile
    ds = ds.rio.clip([row['geometry']])

    # Get coverage of the tile
    intertidal_coverage_sdb = ds.notnull().sum() / ds.size * 100
    gdf_tiles.at[i, 'intertidal_coverage_sdb'] = intertidal_coverage_sdb
    gdf_tiles.at[i, 'intertidal_area_sdb'] = gdf_tiles.at[i, 'tile_area'] * intertidal_coverage_sdb/100

100%|██████████| 1180/1180 [03:16<00:00,  5.99it/s]


### 7. Sum global coverage per region

In [28]:
# Group gdf by ref_region
gdf_tiles_grouped = gdf_tiles[['ref_region', 'tile_area', 'intertidal_area_ed', 'intertidal_area_sdb']].groupby('ref_region').agg(
    {'tile_area': 'sum', 'intertidal_area_ed': 'sum', 'tile_area': 'sum', 'intertidal_area_sdb': 'mean'}).reset_index()
gdf_tiles_grouped['tile_area'] = gdf_tiles_grouped['tile_area'].astype(int)
gdf_tiles_grouped['intertidal_area_ed'] = gdf_tiles_grouped['intertidal_area_ed'].astype(int)
gdf_tiles_grouped['intertidal_area_sdb'] = gdf_tiles_grouped['intertidal_area_sdb'].astype(int)
gdf_tiles_grouped['intertidal_area_sdb'] = gdf_tiles_grouped['intertidal_area_sdb'].round(2)

gdf_tiles_grouped

,ref_region,tile_area,intertidal_area_ed,intertidal_area_sdb
0,CAF,27568,900,7
1,CAU,41353,2129,9
2,EAO,7658,231,13
3,EAU,64327,2448,8
4,EIO,35226,829,1
5,ESAF,65859,2995,16
6,MDG,117933,5300,13
7,MED,188387,9215,20
8,NAU,168476,7775,10
9,NEAF,15316,413,16
